# Démonstration Cartomize — Sentinel-2 L2A réel

**Objectif :** partir de vraies bandes téléchargées, contrôler leur provenance, lancer le prétraitement automatique, calculer des indices spectraux et produire des statistiques vérifiables.

Aucune image synthétique et aucun résultat prédéfini ne sont utilisés. L'identifiant exact de la scène est lu dans `data/provenance.json`.

> L'emprise est une fenêtre d'analyse située à Kinshasa et non une limite administrative.

## 1. Préparer l'environnement

Le dossier `data` est produit par `download_real_data.py`. Dans la version validée sur GitHub, les fichiers téléchargés et leurs SHA-256 sont conservés avec le tutoriel.

In [1]:
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import cartomize as cm

ROOT = Path.cwd()
if ROOT.name != "demonstration-carto":
    candidate = ROOT / "demonstration-carto"
    if candidate.is_dir():
        ROOT = candidate
DATA = ROOT / "data"
RESULTS = ROOT / "results"
if not (DATA / "provenance.json").is_file():
    raise FileNotFoundError("Données réelles absentes : exécuter download_real_data.py avant le notebook.")
if RESULTS.exists():
    shutil.rmtree(RESULTS)
RESULTS.mkdir()
provenance = json.loads((DATA / "provenance.json").read_text(encoding="utf-8"))
print("Scène :", provenance["selected_item_id"])
print("Date :", provenance["selected_datetime"])
print("Plateforme :", provenance["platform"])
print("Tuile MGRS :", provenance["mgrs_tile"])
print("Nuages déclarés :", provenance["global_cloud_cover_percent"], "%")
print("Pixels SCL utilisables sur l'emprise :", round(provenance["aoi_scl_usable_percent_before_processing"], 2), "%")

Scène : S2B_33MWR_20240720_0_L2A
Date : 2024-07-20T09:21:50.068000Z
Plateforme : sentinel-2b
Tuile MGRS : None
Nuages déclarés : 9.286346 %
Pixels SCL utilisables sur l'emprise : 99.95 %


## 2. Contrôler les fichiers avant traitement

On vérifie le CRS, la résolution, le type de pixel et les paramètres de calibration réellement déclarés dans le manifeste Cartomize.

In [2]:
scene = cm.load_scenes_manifest(DATA / "scenes.json")[0]
rows = []
for name, band in scene.bands.items():
    with rasterio.open(band.path) as src:
        rows.append({
            "bande": name,
            "fichier": Path(band.path).name,
            "largeur": src.width,
            "hauteur": src.height,
            "crs": str(src.crs),
            "resolution": src.res,
            "dtype": src.dtypes[0],
            "scale": band.scale,
            "offset": band.offset,
            "unite": band.unit,
        })
display(pd.DataFrame(rows))
print("Masque qualité :", Path(scene.quality).name, "|", scene.quality_kind)
print("Notes de calibration :")
for note in scene.notes:
    print("-", note)

,bande,fichier,largeur,hauteur,crs,resolution,dtype,scale,offset,unite
0,blue,B02.tif,667,664,EPSG:32733,"(10.0, 10.0)",uint16,0.0001,0.0,reflectance
1,green,B03.tif,667,664,EPSG:32733,"(10.0, 10.0)",uint16,0.0001,0.0,reflectance
2,red,B04.tif,667,664,EPSG:32733,"(10.0, 10.0)",uint16,0.0001,0.0,reflectance
3,nir,B08.tif,667,664,EPSG:32733,"(10.0, 10.0)",uint16,0.0001,0.0,reflectance


Masque qualité : SCL.tif | sentinel_scl
Notes de calibration :
- Earth Search COG harmonisé blue : scale déclaré=0.0001, offset déclaré=-0.1, offset effectif=0 ; earthsearch:boa_offset_applied=true, aucune seconde application du décalage BOA.
- Earth Search COG harmonisé green : scale déclaré=0.0001, offset déclaré=-0.1, offset effectif=0 ; earthsearch:boa_offset_applied=true, aucune seconde application du décalage BOA.
- Earth Search COG harmonisé red : scale déclaré=0.0001, offset déclaré=-0.1, offset effectif=0 ; earthsearch:boa_offset_applied=true, aucune seconde application du décalage BOA.
- Earth Search COG harmonisé nir : scale déclaré=0.0001, offset déclaré=-0.1, offset effectif=0 ; earthsearch:boa_offset_applied=true, aucune seconde application du décalage BOA.


## 3. Prétraitement automatique

Cartomize contrôle la couverture, applique la calibration déclarée, utilise la couche SCL, harmonise les grilles et découpe sur la zone d'étude. Le seuil de démonstration est fixé à 95 % avant de connaître le résultat final.

In [3]:
prepared = cm.preprocess_imagery(
    DATA,
    RESULTS / "preprocessed",
    aoi=DATA / "aoi.geojson",
    band_order=["blue", "green", "red", "nir"],
    coverage_threshold=95,
    allow_partial=False,
    mask_clouds=True,
    mask_saturation=False,
    multiband=True,
    separate_bands=True,
    composition="natural",
)
print("Statut :", prepared.report["status"])
print("Scènes retenues :", prepared.report["selected_scene_ids"])
print("Couverture utile :", round(prepared.report["usable_percent"], 3), "%")
print("Mosaïque :", prepared.report["mosaic"])
print("Rapport :", prepared.manifest)
product = prepared.products[0]
display(pd.DataFrame([product]))

Statut : partial
Scènes retenues : ['S2B_33MWR_20240720_0_L2A']
Couverture utile : 99.953 %
Mosaïque : False
Rapport : /home/runner/work/cartomize-python/cartomize-python/demonstration-carto/results/preprocessed/preprocessing.json


,scenes,multiband,bands,composition,source_index,report
0,[S2B_33MWR_20240720_0_L2A],/home/runner/work/cartomize-python/cartomize-p...,"{'1': {'name': 'blue', 'path': '/home/runner/w...",/home/runner/work/cartomize-python/cartomize-p...,/home/runner/work/cartomize-python/cartomize-p...,/home/runner/work/cartomize-python/cartomize-p...


## 4. Composition en couleurs naturelles

La composition est un produit de visualisation. Les analyses suivantes utilisent toujours le raster scientifique multibande.

In [4]:
composition = Path(product["composition"])
with rasterio.open(composition) as src:
    rgb = np.moveaxis(src.read()[:3], 0, -1)
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(rgb)
ax.set_title("Sentinel-2 L2A — composition naturelle après masque et découpage")
ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "natural_color.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Choisir les indices à calculer

Les indices ne sont **pas imposés** par le workflow. L'utilisateur choisit ceux qui correspondent à son objectif et aux bandes réellement disponibles.

Avec les bandes Sentinel-2 B02, B03, B04 et B08 présentes dans cette démonstration, Cartomize peut calculer plusieurs indices sans ajouter de bande inexistante. La cellule suivante identifie automatiquement les indices compatibles, puis en sélectionne plusieurs pour la démonstration.

Pour un indice nécessitant SWIR ou red-edge, Cartomize doit disposer des bandes correspondantes ; le tutoriel ne les invente pas.

In [5]:
available_bands = {"blue", "green", "red", "nir"}
catalog = pd.DataFrame(cm.list_indices())
catalog["compatible"] = catalog["bands"].apply(lambda bands: set(bands).issubset(available_bands))
compatible = catalog.loc[catalog["compatible"], ["name", "title", "bands", "formula", "domain", "reference"]].reset_index(drop=True)
display(compatible)

# Choix de l'utilisateur pour cette démonstration.
# Modifier simplement cette liste pour calculer un sous-ensemble.
selected_indices = [
    "NDVI", "NDWI", "EVI", "EVI2", "SAVI",
    "OSAVI", "MSAVI", "GNDVI", "ARVI", "VARI", "SR",
]
missing = sorted(set(selected_indices) - set(compatible["name"]))
if missing:
    raise ValueError(f"Indices incompatibles avec les bandes disponibles : {missing}")
print("Indices sélectionnés :", ", ".join(selected_indices))

,name,title,bands,formula,domain,reference
0,NDVI,Indice de végétation par différence normalisée,"(nir, red)",(nir-red)/(nir+red),Végétation,https://ntrs.nasa.gov/citations/19740022614
1,EVI,Indice de végétation amélioré,"(nir, red, blue)",g*(nir-red)/(nir+C1*red-C2*blue+L),Végétation,https://doi.org/10.1016/S0034-4257(96)00112-5
2,EVI2,Indice de végétation amélioré à deux bandes,"(nir, red)",g*(nir-red)/(nir+2.4*red+L),Végétation,https://doi.org/10.1016/j.rse.2008.06.006
3,SAVI,Indice de végétation ajusté au sol,"(nir, red)",(1+L)*(nir-red)/(nir+red+L),Végétation,https://doi.org/10.1016/0034-4257(88)90106-X
4,OSAVI,Indice de végétation ajusté au sol optimisé,"(nir, red)",(nir-red)/(nir+red+0.16),Végétation,https://doi.org/10.1016/0034-4257(95)00186-7
5,MSAVI,Indice de végétation ajusté au sol modifié,"(nir, red)",(2*nir+1-sqrt((2*nir+1)**2-8*(nir-red)))/2,Végétation,https://doi.org/10.1016/0034-4257(94)90134-1
6,GNDVI,Indice de végétation normalisé utilisant le vert,"(nir, green)",(nir-green)/(nir+green),Végétation,https://doi.org/10.1016/S0034-4257(96)00072-7
7,NDWI,Indice d’eau par différence normalisée (McFeet...,"(green, nir)",(green-nir)/(green+nir),Eau,https://doi.org/10.1080/01431169608948714
8,ARVI,Indice de végétation résistant aux effets atmo...,"(nir, red, blue)",(nir-(red-gamma*(red-blue)))/(nir+(red-gamma*(...,Végétation,https://doi.org/10.1109/36.134076
9,VARI,Indice de végétation visible résistant aux eff...,"(green, red, blue)",(green-red)/(green+red-blue),Végétation,https://doi.org/10.1016/S0034-4257(01)00289-9


Indices sélectionnés : NDVI, NDWI, EVI, EVI2, SAVI, OSAVI, MSAVI, GNDVI, ARVI, VARI, SR


## 6. Calcul automatique des indices et statistiques

Cartomize calcule tous les indices sélectionnés dans **un seul raster multibande**. Les statistiques sont ensuite dérivées directement des pixels valides de chaque bande de résultat.

Les valeurs ci-dessous ne sont pas saisies à l'avance : elles sont recalculées à chaque exécution à partir de la scène Sentinel-2 L2A réellement traitée.

In [6]:
indices_path = cm.spectral_indices(
    product["multiband"],
    RESULTS / "indices.tif",
    indices=selected_indices,
)

index_stats = {}
with rasterio.open(indices_path) as src:
    if tuple(src.descriptions) != tuple(selected_indices):
        raise RuntimeError("L'ordre des bandes d'indices ne correspond pas au choix demandé.")
    n = len(selected_indices)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.6 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for band, name in enumerate(src.descriptions, 1):
        data = src.read(band, masked=True).astype("float64")
        valid = data.compressed()
        if not valid.size:
            raise RuntimeError(f"Aucun pixel valide pour {name}.")
        index_stats[name] = {
            "count": int(valid.size),
            "min": float(valid.min()),
            "max": float(valid.max()),
            "mean": float(valid.mean()),
            "std": float(valid.std()),
            "p05": float(np.quantile(valid, .05)),
            "p50": float(np.quantile(valid, .50)),
            "p95": float(np.quantile(valid, .95)),
        }
        ax = axes[band - 1]
        view = ax.imshow(data, cmap="RdYlGn")
        ax.set_title(name)
        ax.axis("off")
        fig.colorbar(view, ax=ax, shrink=.72)
    for ax in axes[n:]:
        ax.axis("off")
    fig.savefig(RESULTS / "indices_overview.png", dpi=180, bbox_inches="tight")

stats_df = pd.DataFrame(index_stats).T
stats_df.to_csv(RESULTS / "indices_stats.csv", encoding="utf-8-sig")
display(stats_df)
plt.show()

,count,min,max,mean,std,p05,p50,p95
NDVI,441823.0,-0.373441,0.826395,0.090253,0.186505,-0.161707,0.113782,0.430157
NDWI,441823.0,-0.998533,0.375274,-0.220408,0.175803,-0.519501,-0.248574,0.032505
EVI,441823.0,-0.122345,0.769383,0.053566,0.085229,-0.039066,0.050113,0.211033
EVI2,441823.0,-0.193374,0.717595,0.052331,0.084642,-0.039827,0.049044,0.209333
SAVI,441823.0,-0.220111,0.659289,0.056176,0.090524,-0.046935,0.055257,0.224690
OSAVI,441823.0,-0.186679,0.641556,0.063636,0.109266,-0.069200,0.068369,0.267730
MSAVI,441823.0,-0.224208,0.701716,0.049085,0.078781,-0.034216,0.044482,0.194030
GNDVI,441823.0,-0.375274,0.998533,0.220408,0.175803,-0.032505,0.248574,0.519501
ARVI,441823.0,-0.196629,0.967181,0.421377,0.145236,0.209460,0.411989,0.696000
VARI,441823.0,-1.271350,0.679602,-0.187674,0.058825,-0.275545,-0.190123,-0.094142


## 7. Résumé de l'automatisation

Le résultat central de la démonstration n'est pas un indice particulier : c'est la chaîne automatique.

**Bandes téléchargées → contrôle de cohérence → contrôle de couverture → masque qualité → découpage sur la zone d'étude → mosaïque si plusieurs scènes sont nécessaires → raster scientifique multibande → composite → indices choisis → statistiques.**

Le rapport ci-dessous conserve la décision prise par Cartomize et les produits effectivement générés.

In [7]:
metrics = {
    "schema": "cartomize.demonstration.automation.v2",
    "scene": {
        "id": provenance["selected_item_id"],
        "datetime": provenance["selected_datetime"],
        "platform": provenance["platform"],
        "global_cloud_cover_percent": provenance["global_cloud_cover_percent"],
    },
    "preprocessing": {
        "status": prepared.report["status"],
        "usable_percent": float(prepared.report["usable_percent"]),
        "threshold_met": bool(prepared.report["threshold_met"]),
        "selected_scene_ids": prepared.report["selected_scene_ids"],
        "mosaic": bool(prepared.report["mosaic"]),
        "band_order": prepared.report["parameters"]["band_order"],
        "resolution_m": prepared.report["parameters"]["resolution_m"],
        "crs": prepared.report["parameters"]["crs"],
    },
    "selected_indices": selected_indices,
    "indices": index_stats,
    "products": {
        "multiband": str(Path(product["multiband"]).relative_to(RESULTS / "preprocessed")),
        "composition": str(Path(product["composition"]).relative_to(RESULTS / "preprocessed")),
        "indices": "indices.tif",
        "statistics": "indices_stats.csv",
        "overview": "indices_overview.png",
    },
}
(RESULTS / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
display(pd.DataFrame([metrics["preprocessing"]]))

,status,usable_percent,threshold_met,selected_scene_ids,mosaic,band_order,resolution_m,crs
0,partial,99.952944,True,[S2B_33MWR_20240720_0_L2A],False,"[blue, green, red, nir]",10.0,"PROJCS[""WGS 84 / UTM zone 33S"",GEOGCS[""WGS 84""..."


## 8. Classification : seulement si l'utilisateur fournit des ROI

La classification n'est pas exécutée dans cette démonstration.

Elle reste disponible comme étape **optionnelle** : l'utilisateur peut créer des régions d'intérêt (ROI), les exporter avec leurs classes, puis fournir ces échantillons à Cartomize pour l'apprentissage et la classification. Pour une évaluation scientifique, un jeu de validation indépendant doit être fourni.

Ainsi, la démonstration principale reste centrée sur l'automatisation du prétraitement et des analyses raster.

## 9. Refaire le même parcours dans l'interface graphique

Le guide [GUIDE_INTERFACE.md](GUIDE_INTERFACE.md) reprend exactement ce parcours dans l'application de bureau : contrôle automatique, découpage, mosaïque lorsque nécessaire, composite, choix des indices et statistiques.